# Counting Slides from a YAML File
In this notebook, we will count the total number of slides from `.pptx` files listed in Zenodo records retrieved from a YAML file.

## Step 1: Load YAML File
We will start by loading the YAML file to extract the URLs.

In [1]:
import yaml
import os

yaml_path = "../resources/scadsai.yml"

with open(yaml_path, "r", encoding="utf-8") as yml_file:
    data = yaml.safe_load(yml_file)

urls = []
records = data["resources"]

for r in records:
    r_urls = r["url"]
    if not isinstance(r_urls, list):
        r_urls = [r_urls]
    for u in r_urls:
        urls.append(u)

len(urls)

306

In [2]:
urls[:3]

['https://zenodo.org/records/17541532',
 'https://doi.org/10.5281/zenodo.17541532',
 'https://zenodo.org/records/17478160']

## Step 2: Identify Zenodo URLs and Fetch Record Information via API
For URLs containing `https://zenodo.org`, retrieve the file details via the Zenodo API.

In [3]:
import requests

# Filter for Zenodo URLs
zenodo_urls = [url for url in urls if "https://zenodo.org" in url]
len(zenodo_urls), zenodo_urls[0]

(74, 'https://zenodo.org/records/17541532')

In [4]:
# Fetch record information
def get_zenodo_files(zenodo_url):
    record_id = zenodo_url.split("/")[-1]
    api_url = f"https://zenodo.org/api/records/{record_id}"
    response = requests.get(api_url)
    return response.json().get("files", [])

all_files = [get_zenodo_files(url) for url in zenodo_urls]
len(all_files)

74

In [5]:
all_files[:3]

[[{'id': '9b8e64aa-7261-423c-b1ac-9c2a07b9aada',
   'key': '2025-11-06_Poster_All-Hands-Meeting_DFKI.pdf',
   'size': 7803030,
   'checksum': 'md5:9110d519ac12110aebb9a3de5fc546fe',
   'links': {'self': 'https://zenodo.org/api/records/17541532/files/2025-11-06_Poster_All-Hands-Meeting_DFKI.pdf/content'}}],
 [],
 [{'id': '9ff95f15-3da7-49a2-8fc0-f0b95be02a2f',
   'key': 'GenAI_ArbStud_v4.pptx',
   'size': 64911577,
   'checksum': 'md5:da1eb0dedf8937814154ddd7d38157e6',
   'links': {'self': 'https://zenodo.org/api/records/17422917/files/GenAI_ArbStud_v4.pptx/content'}},
  {'id': '9687f1c3-69cc-4939-9123-be6f03728d21',
   'key': 'Leipzig_University.docx',
   'size': 123611,
   'checksum': 'md5:495736b1ae10ccd3a6debd0bc462571b',
   'links': {'self': 'https://zenodo.org/api/records/17422917/files/Leipzig_University.docx/content'}},
  {'id': 'e669420a-9ff3-4037-80ca-9262119ef2b9',
   'key': 'GenAI_ArbStud_v4.pdf',
   'size': 5454478,
   'checksum': 'md5:7d0f87ef7cc0f255cd248ebd957b9826',
   

## Step 3: Filter `.pptx` Files and Download Them
Check the files in each Zenodo record and download those with `.pptx` extensions.

In [6]:
import os

# Ensure temp directory exists
os.makedirs("./temp", exist_ok=True)

# Filter and download .pptx files
pptx_files = []
for files in all_files:
    for file in files:
        if file["key"].endswith(".pptx"):
            response = requests.get(file["links"]["self"])
            file_path = f"./temp/{file['key']}"
            with open(file_path, "wb") as f:
                f.write(response.content)
            pptx_files.append(file_path)

pptx_files

['./temp/GenAI_ArbStud_v4.pptx',
 './temp/KIKT1_Intro.pptx',
 './temp/KIKT3_ResponsibleAI.pptx',
 './temp/KIKT2_Advanced.pptx',
 './temp/2.1.RDM.pptx',
 './temp/20250915_dataliteracy-rdm-coding_final.pptx',
 './temp/2.4.Explainbility.pptx',
 './temp/2025-09_GA_coding-effectivly-with-AI.pptx',
 './temp/How_LLMs_impact_BIDS_v2.pptx',
 './temp/Repro_BIA_PNAI.pptx',
 './temp/LLMs_BIA_v32.pptx',
 './temp/AAISDA_v1.pptx',
 './temp/GenAI_ArbStud_v3.pptx',
 './temp/07_Deep_Learning.pptx',
 './temp/05_GPUs_Tiles_QA.pptx',
 './temp/04_Surface_Reconstruction_Feature_Extraction.pptx',
 './temp/02_Image_Processing.pptx',
 './temp/01_Introduction_BIDS_2025.pptx',
 './temp/03_Image_segmentation.pptx',
 './temp/06_Sup_Unsup_Machine_Learning.pptx',
 './temp/09_Code_Generation.pptx',
 './temp/08_LLMs_Intro.pptx',
 './temp/11_RDM.pptx',
 './temp/10_MultiModal_LMs.pptx',
 './temp/12_Summary_BIDS_2025.pptx',
 './temp/Haase_LLMs_BIA_Helmholtz_Imaging.pptx',
 './temp/GenAI_ArbStud.pptx',
 './temp/EU_AI_Act_e

## Step 4: Count Slides in Downloaded `.pptx` Files
Use the `python-pptx` library to open each `.pptx` file and count the slides.

In [8]:
from pptx import Presentation

# Count slides
def count_slides(file_path):
    presentation = Presentation(file_path)
    return len(presentation.slides)

total_slides = sum(count_slides(file_path) for file_path in pptx_files)
total_slides

3516